In [ ]:
!pip install datasets[audio] librosa transformers pytorch-metric-learning

In [ ]:
from google.colab import drive, runtime

import sys
import glob
import random
import pickle
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import StratifiedKFold
from datasets import load_from_disk, concatenate_datasets, Audio, disable_caching

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn import Module, Linear
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Sampler, DataLoader
from torchvision.models import resnet18

from pytorch_metric_learning import losses

NUM_EPOCHS = 6
NUM_FOLDS = 3
NUM_CLASSES = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 1e-4
SEED = 42

In [ ]:
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/ASP/Project/speaker_chunks"

In [ ]:
def seed_everything(seed=42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # PyTorch GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # CuDNN
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

# Data Preprocessing


Load data

In [ ]:
# Disable caching so no additional files are saved in drive
disable_caching()

In [ ]:
drive_path = "./chunk_*"
chunk_paths = sorted(glob.glob(drive_path))

chunks = []
for path in chunk_paths:
    ds = load_from_disk(path)
    ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
    chunks.append(ds)

full_dataset = concatenate_datasets(chunks)

In [ ]:
# Fixed held-out test set
split = full_dataset.train_test_split(test_size=0.1, seed=42)
trainval_dataset = split["train"] # For cross validation
test_dataset = split["test"] # Fixed

# Get the labels for stratification
labels = np.array(trainval_dataset["label"])

# 9-way stratified split, take 3 folds
skf = StratifiedKFold(n_splits=9, shuffle=True, random_state=42)
fold_indices = list(skf.split(np.zeros(len(labels)), labels))

folds = [] # Collect the 3 folds
for i in range(NUM_FOLDS):
    train_idx, dev_idx = fold_indices[i]
    train_dataset = trainval_dataset.select(train_idx.tolist())
    dev_dataset = trainval_dataset.select(dev_idx.tolist())
    folds.append((train_dataset, dev_dataset))
    print(f"Fold {i}: train={len(train_dataset)}, dev={len(dev_dataset)}, test={len(test_dataset)}")

Prepocess data using the Dataset_Builder class

In [ ]:
# Add specific folder containing dataset_builder.py to path
sys.path.append("./contrastive/files/utils")

from dataset_builder import Dataset_Builder

In [ ]:
# Training sets
config = {"shortest_duration": 4.0, "timesteps": 400} # (optional)
dataset_train_folds = []

# Loop for 3 folds
for fold in folds:
  train_builder = Dataset_Builder(fold[0], **config)
  train_builder.filter()
  train_builder.preprocess()
  dataset_train_folds.append(train_builder.get_dataset())

In [ ]:
# Development sets
dataset_dev_folds = []

# Loop for 3 folds
for fold in folds:
  dev_builder = Dataset_Builder(fold[1], **config)
  dev_builder.filter()
  dev_builder.preprocess()
  dataset_dev_folds.append(dev_builder.get_dataset())

#Create DataLoaders

In [ ]:
# Create spk2indxs for given fold
def get_spk2indxs(fold):
    spk2indxs = defaultdict(list)
    for i, item in tqdm(enumerate(fold), total=len(fold)):
        spk2indxs[item["label"]].append(i)

    return dict(spk2indxs)

Define contrastive sampler

In [ ]:
# Batch size of N*E=64 (increased compared to Cross-Entropy method)
N = 8
E = 8

class ContrastiveSampler(Sampler):
  def __init__(self, dataset, spk2indxs, num_spk_per_batch, num_ex_per_spk):
    self.dataset = dataset
    self.spk2indxs = spk2indxs
    self.N = num_spk_per_batch
    self.E = num_ex_per_spk
    self.bs = self.N * self.E

    for k, v in self.spk2indxs.items():
      if len(v) < self.E:
        raise ValueError(f"Speaker {k} has less than {self.E} examples")

  def __len__(self):
    return len(self.dataset) // self.bs

  def __iter__(self):
    num_batches = len(self.dataset) // self.bs

    for _ in range(num_batches):
      batch_indxs = []

      sampled_spk = random.sample(list(self.spk2indxs.keys()), self.N)
      for spk in sampled_spk:
        selected_samples = random.sample(self.spk2indxs[spk], self.E)
        batch_indxs.extend(selected_samples)

      yield batch_indxs

Construct dataloader

In [ ]:
def collate_fn(batch):
    log_mels = torch.stack([b["log_mel"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch])
    return (log_mels, labels)

dataloader_train_folds = []
dataloader_dev_folds = []

for train_dataset, dev_dataset in zip(dataset_train_folds, dataset_dev_folds):

  spk2indxs_train = get_spk2indxs(train_dataset)
  spk2indxs_dev = get_spk2indxs(dev_dataset)

  train_sampler = ContrastiveSampler(train_dataset, spk2indxs_train, N, E)
  dev_sampler = ContrastiveSampler(dev_dataset, spk2indxs_dev, N, E)

  dataloader_train = DataLoader(
      train_dataset,
      batch_sampler=train_sampler,
      shuffle=False,
      collate_fn=collate_fn,
      pin_memory=True
  )
  dataloader_dev = DataLoader(
      dev_dataset,
      batch_sampler=dev_sampler,
      shuffle=False,
      collate_fn=collate_fn,
      pin_memory=True
  )

  dataloader_train_folds.append(dataloader_train)
  dataloader_dev_folds.append(dataloader_dev)

# Training

Define model and loss

In [ ]:
class ContrastiveModel(Module):
  def __init__(self, base_model, embed_dim):
    super().__init__()
    self.model = base_model
    self.model.fc = Linear(in_features=512, out_features=embed_dim, bias=True)

  def forward(self, x):
    x = self.model(x)
    return nn.functional.normalize(x, dim=-1) # Avoid representation collapse

In [ ]:
con_loss = losses.NTXentLoss(temperature=0.07)

In [ ]:
# Initialize tensorboard for logging
%load_ext tensorboard
%tensorboard --logdir "resnet_contr_cross_val_6e_4s/runs"

Training loop

In [ ]:
for fold_idx, (dataloader_train, dataloader_dev) in enumerate(zip(dataloader_train_folds, dataloader_dev_folds)):
  print(f"\n=========== FOLD {fold_idx + 1} ===========")

  # Re-initialize everything
  best = float("inf")
  base_model = resnet18(weights="DEFAULT")
  model = ContrastiveModel(base_model, NUM_CLASSES)
  model.to(DEVICE)
  optimizer = AdamW(params=model.parameters(), lr=LR)
  writer = SummaryWriter(log_dir=f"resnet_contr_cross_val_6e_4s/runs/fold_{fold_idx + 1}") # Separate TB log per fold
  best_ckpt_path = f"resnet_contr_cross_val_6e_4s/best_speaker_resnet_fold{fold_idx + 1}.pth"

  for epoch in range(1, NUM_EPOCHS+1):
    print(f"--- Epoch: {epoch} ---")

    model.train()
    for indx, (log_mels, labels) in tqdm(enumerate(dataloader_train), desc="Epoch Progress", total=len(dataloader_train)):
      log_mels = log_mels.to(DEVICE)
      labels = labels.to(DEVICE)

      optimizer.zero_grad()
      embeds = model(log_mels)
      loss = con_loss(embeds, labels)
      loss.backward()
      optimizer.step()

      global_step = (epoch-1)*len(dataloader_train) + indx
      writer.add_scalar("Loss/Train", loss.item(), global_step)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
      for log_mels, labels in tqdm(dataloader_dev, desc="Validation Progress", total=len(dataloader_dev)):
        log_mels = log_mels.to(DEVICE)
        labels = labels.to(DEVICE)

        embeds = model(log_mels)
        loss = con_loss(embeds, labels)

        val_loss += loss.item()
    avg_val_loss = val_loss / len(dataloader_dev)

    print(f"Fold {fold_idx + 1} | Epoch {epoch} finished | Val Loss: {avg_val_loss:.4f}")

    writer.add_scalar("Loss/Val", avg_val_loss, epoch)

    if avg_val_loss < best:
      best = avg_val_loss
      torch.save(model.state_dict(), best_ckpt_path)
      print(f"New best model saved for fold {fold_idx + 1}! (Val Loss: {avg_val_loss:.4f})")

  writer.close()
  print(f"Fold {fold_idx + 1} finished")

Saved spk2indxs to avoid recomputing indexes in the future

In [ ]:
for fold_idx, (train_loader, dev_loader) in enumerate(zip(dataloader_train_folds, dataloader_dev_folds)):

    spk2indxs_train = train_loader.batch_sampler.spk2indxs
    spk2indxs_dev = dev_loader.batch_sampler.spk2indxs

    with open(f"resnet_contr_cross_val_6e_4s/spk2indxs_train_fold_{fold_idx+1}.pkl", "wb") as f:
        pickle.dump(spk2indxs_train, f)

    with open(f"resnet_contr_cross_val_6e_4s/spk2indxs_dev_fold_{fold_idx+1}.pkl", "wb") as f:
        pickle.dump(spk2indxs_dev, f)

print("Saved all fold indices successfully!")

In [ ]:
# Stop colab session
runtime.unassign()